组合计数
1. 基础计数： 处理元素与容器的区分度（球盒模型）- Twelvefold Way
2. 约束计数： 处理容斥原理、抽屉原理（约束条件）。
3. 等价计数： 处理群论下的对称性（Burnside & Pólya）。

### Twelvefold Way -> 二十四路计数模型


| Ball $N$ | Box $K$ | $f$ unrestricted | $f$ one-to-one injective| $f$ onto Surjective|
|---|---|---|---|---|
| labeled | labeled | $k^n$ | $(k-n+1)_n$ | $k!\,S(n,k)$ |
| unlabeled | labeled | $\binom{k+n-1}{n}$ | $\binom{k}{n}$ | $\binom{n-1}{\,n-k\,}$ |
| labeled | unlabeled | $S(n,1)+S(n,2)+\cdots+S(n,k)$ | $\begin{cases}1 & n\le k\\0 & n>k\end{cases}$ | $S(n,k)$ |
| unlabeled | unlabeled | $p_k(n)$ | $\begin{cases}1 & n\le k\\0 & n>k\end{cases}$ | $p_k(n)-p_{k-1}(n)$ |

二十四路计数的三个逻辑维度
- 元素（球）是否可区分？ (Distinguishable / Indistinguishable)
- 容器（盒）是否可区分？ (Distinguishable / Indistinguishable)
- 容器内的元素是否有序？ (Ordered / Unordered) —— 这是 24 路相比 12 路增加的关键维度

ps: Twelvefold Way是所有计数的核心，剩下的“有序”情况(即24路)可以通过“先分组、再排列”推导出来

Structural summary
  - Stirling numbers = unlabeled blocks
  - Bell numbers = all partitions
  - Partitions = size-only structure


球是否相同|盒是否相同|盒内是否有序|无限制 (L)|单射 (I)|满射 (S)
|-|-|-|-|-|-|
不同|不同|无序|m^n|"P(m,n)"|"m!S2​(n,m)"
相同|不同|无序|"C(n+m−1,m−1)"|"C(m,n)"|"C(n−1,m−1)"
不同|相同|无序|"∑S2​(n,k)"|[n≤m]|"S2​(n,m)"
相同|相同|无序|"p(n,m)"|[n≤m]|"p(n−m,m)"
不同|不同|有序|"P(n+m−1,n)"|"P(m,n)"|"n!C(n−1,m−1)"
不同|相同|有序|(分配并除以 m!)|[n≤m]|(同左)

In [ ]:
# Twelvefold Way in SageMath (n balls --> k boxes)
def TF_distinctBalls_labeledBoxes(n, k): #     Balls distinct/labeled, boxes labeled.
    any = k^n                                                 # k^n , unrestricted
    inj = 0 if n > k else factorial(k) // factorial(k - n)    # (k)_n, falling_factorial(k,n)
    sur = factorial(k) * stirling_number2(n, k)               # k! S(n,k)
    return any, inj, sur

def TF_identicalBalls_labeledBoxes(n, k): # Occupancy is vector
    any = binomial(n + k - 1, k - 1)              # xi>=0, sum=n -> stars and bars C(n+k-1,k-1)
    inj = 0 if n > k else binomial(k, n)          # xi in {0,1}, sum=n -> choose n boxes
    sur = 0 if n < k else binomial(n - 1, k - 1)  # xi>=1, sum=n -> compositions count C(n-1,k-1)
    return any, inj, sur

def TF_distinctBalls_unlabeledBoxes(n, k): #   Balls distinct, boxes unlabeled/identical
    any = sum(stirling_number2(n, i) for i in range(0, k+1)) # set partitions into <= k blocks -> sum_{i=0..k} S(n,i)
    inj = 1 if n <= k else 0     # all balls separated (possible iff n<=k)
    sur = stirling_number2(n, k) # set partitions into exactly k blocks -> S(n,k)
    return any, inj, sur

def TF_identicalBalls_unlabeledBoxes(n, k): # Occupancy is a partition (multiset of block sizes)
    any = Partitions(n, max_length=k).cardinality()  # partitions of n into <= k parts -> p_{<=k}(n)
    inj = 1 if n <= k else 0                         # pattern is 1+...+1 (n ones) (possible iff n<=k)
    sur = 0 if n < k else Partitions(n, length=k).cardinality() # partitions of n into k positive parts -> p_k(n)
    return any, inj, sur

def twelvefold_summary(n, k): # print a neat 12-way summary for given n,k
    db_lb = TF_distinctBalls_labeledBoxes(n, k)
    db_ub = TF_distinctBalls_unlabeledBoxes(n, k)
    ib_lb = TF_identicalBalls_labeledBoxes(n, k)
    ib_ub = TF_identicalBalls_unlabeledBoxes(n, k)
    print("")
    print("Twelvefold way (", n, "balls, ",k, "boxes) with capacity rule:")
    print(" Any = unrestricted;  Inj = at most 1 per box;  Sur = no empty box")
    print("")
    print("1) Balls distinct,  Boxes labeled:   Any:", db_lb[0], "  Inj:", db_lb[1], "  Sur:", db_lb[2])
    print("2) Balls distinct,  Boxes unlabeled: Any:", db_ub[0], "  Inj:", db_ub[1], "  Sur:", db_ub[2])
    print("3) Balls identical, Boxes labeled:   Any:", ib_lb[0], "  Inj:", ib_lb[1], "  Sur:", ib_lb[2])
    print("4) Balls identical, Boxes unlabeled: Any:", ib_ub[0], "  Inj:", ib_ub[1], "  Sur:", ib_ub[2])

# Optional: small brute checks (useful for sanity for small n,k)
def brute_distinctBalls_labeledBoxes_any(n, k):
    return k^n      # Count all functions [n]->[k]
def brute_distinctBalls_labeledBoxes_inj(n, k):
    if n > k: return 0      # injections = permutations of k choose n = (k)_n
    return factorial(k) // factorial(k - n)
def brute_distinctBalls_labeledBoxes_sur(n, k):
    return factorial(k) * stirling_number2(n, k) # surj count via Stirling is already exact; brute is expensive

# 3 examples 
twelvefold_summary(5, 5) # it is bijective 
twelvefold_summary(5, 3)
twelvefold_summary(5, 6)


Twelvefold way ( 5 balls,  5 boxes) with capacity rule:
 Any = unrestricted;  Inj = at most 1 per box;  Sur = no empty box

1) Balls distinct,  Boxes labeled:   Any: 3125   Inj: 120   Sur: 120
2) Balls distinct,  Boxes unlabeled: Any: 52   Inj: 1   Sur: 1
3) Balls identical, Boxes labeled:   Any: 126   Inj: 1   Sur: 1
4) Balls identical, Boxes unlabeled: Any: 7   Inj: 1   Sur: 1

Twelvefold way ( 5 balls,  3 boxes) with capacity rule:
 Any = unrestricted;  Inj = at most 1 per box;  Sur = no empty box

1) Balls distinct,  Boxes labeled:   Any: 243   Inj: 0   Sur: 150
2) Balls distinct,  Boxes unlabeled: Any: 41   Inj: 0   Sur: 25
3) Balls identical, Boxes labeled:   Any: 21   Inj: 0   Sur: 6
4) Balls identical, Boxes unlabeled: Any: 5   Inj: 0   Sur: 2

Twelvefold way ( 5 balls,  6 boxes) with capacity rule:
 Any = unrestricted;  Inj = at most 1 per box;  Sur = no empty box

1) Balls distinct,  Boxes labeled:   Any: 7776   Inj: 720   Sur: 0
2) Balls distinct,  Boxes unlabeled: Any: 52 

In [ ]:
# Claude made
def twelvefold_way_table(n, k): # n -- number of balls ;  k -- number of boxes
    results = {}
    print(f"=== THE TWELVEFOLD WAY: {n} balls, {k} boxes ===\n")
    # ========================================
    # CASE 1: Distinguishable balls, Distinguishable boxes, Any number
    # Formula: k^n
    # ========================================
    case1 = k^n
    results['case1'] = case1
    print("Case 1: Distinguishable balls → Distinguishable boxes (unrestricted)")
    print(f"  Formula: k^n = {k}^{n} = {case1}")
    print(f"  Interpretation: Functions from n-set to k-set\n")
    
    # ========================================
    # CASE 2: Distinguishable balls, Distinguishable boxes, Injective (≤1 per box)
    # Formula: k!/(k-n)! = falling_factorial(k,n) if n≤k, else 0
    # ========================================
    if n <= k:
        case2 = falling_factorial(k, n)
    else:
        case2 = 0
    results['case2'] = case2
    print("Case 2: Distinguishable balls → Distinguishable boxes (injective)")
    print(f"  Formula: k!/(k-n)! = P(k,n)")
    print(f"  Value: {case2}")
    print(f"  Interpretation: Injective functions (permutations)\n")
    
    # ========================================
    # CASE 3: Distinguishable balls, Distinguishable boxes, Surjective (≥1 per box)
    # Formula: k! * S(n,k) where S(n,k) is Stirling number of second kind
    # ========================================
    if n >= k:
        case3 = factorial(k) * stirling_number2(n, k)
    else:
        case3 = 0
    results['case3'] = case3
    print("Case 3: Distinguishable balls → Distinguishable boxes (surjective)")
    print(f"  Formula: k! × S(n,k)")
    print(f"  Value: {case3}")
    print(f"  Interpretation: Surjective functions (onto)\n")
    
    # ========================================
    # CASE 4: Distinguishable balls, Indistinguishable boxes, Any number
    # Formula: B(n) where B(n) is the nth Bell number = sum of S(n,j) for j=0..n
    # ========================================
    case4 = bell_number(n)
    results['case4'] = case4
    print("Case 4: Distinguishable balls → Indistinguishable boxes (unrestricted)")
    print(f"  Formula: B_n (Bell number)")
    print(f"  Value: {case4}")
    print(f"  Interpretation: Set partitions of n elements\n")
    
    # ========================================
    # CASE 5: Distinguishable balls, Indistinguishable boxes, Injective
    # Formula: 1 if n≤k, else 0
    # ========================================
    case5 = 1 if n <= k else 0
    results['case5'] = case5
    print("Case 5: Distinguishable balls → Indistinguishable boxes (injective)")
    print(f"  Formula: 1 if n≤k, else 0")
    print(f"  Value: {case5}")
    print(f"  Interpretation: At most one ball per box\n")
    
    # ========================================
    # CASE 6: Distinguishable balls, Indistinguishable boxes, Surjective
    # Formula: S(n,k) - Stirling number of second kind
    # ========================================
    if n >= k and k > 0:
        case6 = stirling_number2(n, k)
    else:
        case6 = 0
    results['case6'] = case6
    print("Case 6: Distinguishable balls → Indistinguishable boxes (surjective)")
    print(f"  Formula: S(n,k) (Stirling number of 2nd kind)")
    print(f"  Value: {case6}")
    print(f"  Interpretation: Partitions of n-set into k non-empty subsets\n")
    
    # ========================================
    # CASE 7: Indistinguishable balls, Distinguishable boxes, Any number
    # Formula: C(n+k-1, n) = C(n+k-1, k-1)
    # ========================================
    case7 = binomial(n + k - 1, n)
    results['case7'] = case7
    print("Case 7: Indistinguishable balls → Distinguishable boxes (unrestricted)")
    print(f"  Formula: C(n+k-1, n)")
    print(f"  Value: {case7}")
    print(f"  Interpretation: Multisets of size n from k elements\n")
    
    # ========================================
    # CASE 8: Indistinguishable balls, Distinguishable boxes, Injective
    # Formula: C(k, n) if n≤k, else 0
    # ========================================
    if n <= k:
        case8 = binomial(k, n)
    else:
        case8 = 0
    results['case8'] = case8
    print("Case 8: Indistinguishable balls → Distinguishable boxes (injective)")
    print(f"  Formula: C(k,n) if n≤k")
    print(f"  Value: {case8}")
    print(f"  Interpretation: n-subsets of k-set\n")
    
    # ========================================
    # CASE 9: Indistinguishable balls, Distinguishable boxes, Surjective
    # Formula: C(n-1, k-1) if n≥k, else 0
    # ========================================
    if n >= k and k > 0:
        case9 = binomial(n - 1, k - 1)
    else:
        case9 = 0
    results['case9'] = case9
    print("Case 9: Indistinguishable balls → Distinguishable boxes (surjective)")
    print(f"  Formula: C(n-1, k-1) if n≥k")
    print(f"  Value: {case9}")
    print(f"  Interpretation: Compositions of n into k positive parts\n")
    
    # ========================================
    # CASE 10: Indistinguishable balls, Indistinguishable boxes, Any number
    # Formula: p(n,k) - number of partitions of n into at most k parts
    # ========================================
    case10 = Partitions(n, max_length=k).cardinality()
    results['case10'] = case10
    print("Case 10: Indistinguishable balls → Indistinguishable boxes (unrestricted)")
    print(f"  Formula: p(n,k) - partitions of n into ≤k parts")
    print(f"  Value: {case10}")
    print(f"  Interpretation: Integer partitions\n")
    
    # ========================================
    # CASE 11: Indistinguishable balls, Indistinguishable boxes, Injective
    # Formula: 1 if n≤k, else 0
    # ========================================
    case11 = 1 if n <= k else 0
    results['case11'] = case11
    print("Case 11: Indistinguishable balls → Indistinguishable boxes (injective)")
    print(f"  Formula: 1 if n≤k, else 0")
    print(f"  Value: {case11}")
    print(f"  Interpretation: At most one ball per box\n")
    
    # ========================================
    # CASE 12: Indistinguishable balls, Indistinguishable boxes, Surjective
    # Formula: p_k(n) - number of partitions of n into exactly k parts
    # ========================================
    if n >= k and k > 0:
        case12 = Partitions(n, length=k).cardinality()
    else:
        case12 = 0
    results['case12'] = case12
    print("Case 12: Indistinguishable balls → Indistinguishable boxes (surjective)")
    print(f"  Formula: p_k(n) - partitions of n into exactly k parts")
    print(f"  Value: {case12}")
    print(f"  Interpretation: Integer partitions of n into k parts\n")
    
    return results


def create_summary_table(n, k):
    r"""
    Create a formatted summary table of all twelve cases.
    """
    results = twelvefold_way_table(n, k)
    
    print("\n" + "="*80)
    print("SUMMARY TABLE")
    print("="*80)
    print(f"{'Case':<6} {'Balls':<18} {'Boxes':<18} {'Restriction':<15} {'Count':<10}")
    print("-"*80)
    
    cases = [
        (1, "Distinguishable", "Distinguishable", "Any", results['case1']),
        (2, "Distinguishable", "Distinguishable", "Injective", results['case2']),
        (3, "Distinguishable", "Distinguishable", "Surjective", results['case3']),
        (4, "Distinguishable", "Indistinguishable", "Any", results['case4']),
        (5, "Distinguishable", "Indistinguishable", "Injective", results['case5']),
        (6, "Distinguishable", "Indistinguishable", "Surjective", results['case6']),
        (7, "Indistinguishable", "Distinguishable", "Any", results['case7']),
        (8, "Indistinguishable", "Distinguishable", "Injective", results['case8']),
        (9, "Indistinguishable", "Distinguishable", "Surjective", results['case9']),
        (10, "Indistinguishable", "Indistinguishable", "Any", results['case10']),
        (11, "Indistinguishable", "Indistinguishable", "Injective", results['case11']),
        (12, "Indistinguishable", "Indistinguishable", "Surjective", results['case12']),
    ]
    
    for case in cases:
        print(f"{case[0]:<6} {case[1]:<18} {case[2]:<18} {case[3]:<15} {case[4]:<10}")
    
    print("="*80)


def demonstrate_specific_cases(n, k):
    r"""
    Demonstrate specific cases with examples using SageMath objects.
    """
    print(f"\n\n=== DETAILED EXAMPLES for n={n}, k={k} ===\n")
    
    # Case 6: Stirling numbers of second kind - show actual partitions
    if n <= 4 and k <= 3:  # Keep examples small
        print(f"Case 6 Example: Partitions of set {{1,...,{n}}} into {k} non-empty subsets:")
        S = SetPartitions(n, k)
        print(f"  Count: {S.cardinality()} = S({n},{k})")
        if S.cardinality() <= 15:
            print(f"  Partitions:")
            for partition in S:
                print(f"    {partition}")
        print()
    
    # Case 10: Integer partitions
    if n <= 10:
        print(f"Case 10 Example: Integer partitions of {n} with at most {k} parts:")
        P = Partitions(n, max_length=k)
        print(f"  Count: {P.cardinality()}")
        if P.cardinality() <= 20:
            print(f"  Partitions:")
            for partition in P:
                print(f"    {partition}")
        print()
    
    # Case 12: Integer partitions with exactly k parts
    if n >= k and n <= 10:
        print(f"Case 12 Example: Integer partitions of {n} into exactly {k} parts:")
        P = Partitions(n, length=k)
        print(f"  Count: {P.cardinality()}")
        if P.cardinality() <= 20:
            print(f"  Partitions:")
            for partition in P:
                print(f"    {partition}")
        print()


# ========================================
# Main execution
# ========================================

# Example 1: Small case
print("EXAMPLE 1: Small case\n")
create_summary_table(4, 3)
demonstrate_specific_cases(4, 3)

print("\n\n" + "#"*80 + "\n\n")

# Example 2: Larger case
print("EXAMPLE 2: Larger case\n")
create_summary_table(7, 5)

print("\n\n" + "#"*80 + "\n\n")

# Example 3: Equal balls and boxes
print("EXAMPLE 3: Equal balls and boxes\n")
create_summary_table(5, 5)

EXAMPLE 1: Small case

=== THE TWELVEFOLD WAY: 4 balls, 3 boxes ===

Case 1: Distinguishable balls → Distinguishable boxes (unrestricted)
  Formula: k^n = 3^4 = 81
  Interpretation: Functions from n-set to k-set

Case 2: Distinguishable balls → Distinguishable boxes (injective)
  Formula: k!/(k-n)! = P(k,n)
  Value: 0
  Interpretation: Injective functions (permutations)

Case 3: Distinguishable balls → Distinguishable boxes (surjective)
  Formula: k! × S(n,k)
  Value: 36
  Interpretation: Surjective functions (onto)

Case 4: Distinguishable balls → Indistinguishable boxes (unrestricted)
  Formula: B_n (Bell number)
  Value: 15
  Interpretation: Set partitions of n elements

Case 5: Distinguishable balls → Indistinguishable boxes (injective)
  Formula: 1 if n≤k, else 0
  Value: 0
  Interpretation: At most one ball per box

Case 6: Distinguishable balls → Indistinguishable boxes (surjective)
  Formula: S(n,k) (Stirling number of 2nd kind)
  Value: 6
  Interpretation: Partitions of n-set 

#### Stirling arise from both permutation & set partitions

```mermaid
flowchart LR
    A["Finite set {1,2,3,4,5}"] --> B[Partition into k blocks]
    A --> C[Permutation]

    %% Set partitions
    B --> B1["Block 1: {1,3,5}"]
    B --> B2["Block 2: {2,4}"]
    B1 --> D["S(n,k)"]
    B2 --> D

    %% Permutation cycles
    C --> C1["Cycle: (1 3 5)"]
    C --> C2["Cycle: (2 4)"]
    C1 --> E["c(n,k)"]
    C2 --> E

    %% Conceptual bridge
    D --> F[Unlabeled components]
    E --> F

    F --> G[Same decomposition into k components]


In [58]:
# Stirling Numbers: arise from Permutations and Set Partitions
# PART 1: from Permutations
print("Definition: The unsigned Stirling number of the first kind s(n,k)")
print("counts the number of permutations of n elements with exactly k cycles.")
def count_cycles(perm):
    """Count the number of cycles in a permutation."""
    n = len(perm)
    visited = [False] * n
    cycle_count = 0
    
    for i in range(n):
        if not visited[i]:
            cycle_count += 1
            j = i
            while not visited[j]:
                visited[j] = True
                j = perm[j] - 1  # Convert to 0-indexed
    
    return cycle_count

def get_cycle_structure(perm):
    """Get the actual cycles of a permutation."""
    n = len(perm)
    visited = [False] * n
    cycles = []
    
    for i in range(n):
        if not visited[i]:
            cycle = []
            j = i
            while not visited[j]:
                visited[j] = True
                cycle.append(j + 1)  # Convert to 1-indexed
                j = perm[j] - 1
            cycles.append(cycle)
    
    return cycles

# Example: Show all permutations of {1,2,3,4} and their cycles
n = 4
print(f"Example: Permutations of {{1,2,3,4}} grouped by cycle count:")
print()

from itertools import permutations as perms

# Group permutations by number of cycles
cycle_groups = {}
for perm in perms(range(1, n+1)):
    num_cycles = count_cycles(perm)
    if num_cycles not in cycle_groups:
        cycle_groups[num_cycles] = []
    cycle_groups[num_cycles].append(perm)

for k in sorted(cycle_groups.keys()):
    print(f"\nPermutations with {k} cycle(s): {len(cycle_groups[k])} permutations")
    print("-" * 60)
    for perm in cycle_groups[k][:5]:  # Show first 5 examples
        cycles = get_cycle_structure(perm)
        print(f"  {perm} → cycles: {cycles}")
    if len(cycle_groups[k]) > 5:
        print(f"  ... and {len(cycle_groups[k]) - 5} more")

print("\n" + "=" * 70)
print("Stirling Numbers of the First Kind (unsigned): s(n,k)")
print("=" * 70)
print()

# Compute using SageMath's built-in function
print("Using SageMath's stirling_number1() function:")
print()
print("  n\\k", end="")
for k in range(1, 7):
    print(f"{k:>8}", end="")
print()
print("-" * 60)

for n in range(1, 8):
    print(f"  {n:2}  ", end="")
    for k in range(1, 7):
        if k <= n:
            s = stirling_number1(n, k)
            print(f"{s:>8}", end="")
        else:
            print(f"{'':>8}", end="")
    print()

print()
print("Verification: s(4,k) matches our permutation count above!")
print()

Definition: The unsigned Stirling number of the first kind s(n,k)
counts the number of permutations of n elements with exactly k cycles.
Example: Permutations of {1,2,3,4} grouped by cycle count:


Permutations with 1 cycle(s): 6 permutations
------------------------------------------------------------
  (2, 3, 4, 1) → cycles: [[1, 2, 3, 4]]
  (2, 4, 1, 3) → cycles: [[1, 2, 4, 3]]
  (3, 1, 4, 2) → cycles: [[1, 3, 4, 2]]
  (3, 4, 2, 1) → cycles: [[1, 3, 2, 4]]
  (4, 1, 2, 3) → cycles: [[1, 4, 3, 2]]
  ... and 1 more

Permutations with 2 cycle(s): 11 permutations
------------------------------------------------------------
  (1, 3, 4, 2) → cycles: [[1], [2, 3, 4]]
  (1, 4, 2, 3) → cycles: [[1], [2, 4, 3]]
  (2, 1, 4, 3) → cycles: [[1, 2], [3, 4]]
  (2, 3, 1, 4) → cycles: [[1, 2, 3], [4]]
  (2, 4, 3, 1) → cycles: [[1, 2, 4], [3]]
  ... and 6 more

Permutations with 3 cycle(s): 6 permutations
------------------------------------------------------------
  (1, 2, 4, 3) → cycles: [[1], [2], [

In [60]:
# PART 2: Stirling Numbers of the Second Kind (from Set Partitions)
print("PART 2: STIRLING NUMBERS OF THE SECOND KIND")
print("Definition: The Stirling number of the second kind S(n,k)")
print("counts the number of ways to partition a set of n elements")
print("into exactly k non-empty subsets.")
print()

# Example: Show all set partitions of {1,2,3,4}
n = 4
print(f"Example: Partitions of {{1,2,3,4}} grouped by number of blocks:")
print()

from sage.combinat.set_partition import SetPartitions

# Group partitions by number of blocks
partition_groups = {}
for partition in SetPartitions(n):
    num_blocks = len(partition)
    if num_blocks not in partition_groups:
        partition_groups[num_blocks] = []
    partition_groups[num_blocks].append(partition)

for k in sorted(partition_groups.keys()):
    print(f"\nPartitions into {k} block(s): {len(partition_groups[k])} partitions")
    print("-" * 60)
    for partition in partition_groups[k]:
        # Format the partition nicely
        blocks = ['{' + ','.join(map(str, sorted(block))) + '}' 
                  for block in partition]
        print(f"  {{{', '.join(blocks)}}}")

print("\n" + "=" * 70)
print("Stirling Numbers of the Second Kind: S(n,k)")
print("=" * 70)
print()

# Compute using SageMath's built-in function
print("Using SageMath's stirling_number2() function:")
print()
print("  n\\k", end="")
for k in range(1, 7):
    print(f"{k:>8}", end="")
print()
print("-" * 60)

for n in range(1, 8):
    print(f"  {n:2}  ", end="")
    for k in range(1, 7):
        if k <= n:
            s = stirling_number2(n, k)
            print(f"{s:>8}", end="")
        else:
            print(f"{'':>8}", end="")
    print()

print()
print("Verification: S(4,k) matches our partition count above!")
print()

PART 2: STIRLING NUMBERS OF THE SECOND KIND
Definition: The Stirling number of the second kind S(n,k)
counts the number of ways to partition a set of n elements
into exactly k non-empty subsets.

Example: Partitions of {1,2,3,4} grouped by number of blocks:


Partitions into 1 block(s): 1 partitions
------------------------------------------------------------
  {{1,2,3,4}}

Partitions into 2 block(s): 7 partitions
------------------------------------------------------------
  {{1,2,3}, {4}}
  {{1,2,4}, {3}}
  {{1,2}, {3,4}}
  {{1,3,4}, {2}}
  {{1,3}, {2,4}}
  {{1,4}, {2,3}}
  {{1}, {2,3,4}}

Partitions into 3 block(s): 6 partitions
------------------------------------------------------------
  {{1,2}, {3}, {4}}
  {{1,3}, {2}, {4}}
  {{1}, {2,3}, {4}}
  {{1,4}, {2}, {3}}
  {{1}, {2,4}, {3}}
  {{1}, {2}, {3,4}}

Partitions into 4 block(s): 1 partitions
------------------------------------------------------------
  {{1}, {2}, {3}, {4}}

Stirling Numbers of the Second Kind: S(n,k)

Using S

In [ ]:
# PART 3: Key Properties and Relationships
print("PART 3: KEY PROPERTIES AND RELATIONSHIPS")
print("=" * 70)
print()

print("1. RECURRENCE RELATIONS")
print("-" * 60)
print()
print("Stirling numbers of the first kind (unsigned):")
print("  s(n,k) = s(n-1,k-1) + (n-1)·s(n-1,k)")
print()
print("Verification for s(5,3):")
n, k = 5, 3
s_nk = stirling_number1(n, k)
s_rec = stirling_number1(n-1, k-1) + (n-1) * stirling_number1(n-1, k)
print(f"  s(5,3) = {s_nk}")
print(f"  s(4,2) + 4·s(4,3) = {stirling_number1(4,2)} + 4·{stirling_number1(4,3)} = {s_rec}")
print(f"  Match: {s_nk == s_rec}")
print()

print("Stirling numbers of the second kind:")
print("  S(n,k) = S(n-1,k-1) + k·S(n-1,k)")
print()
print("Verification for S(5,3):")
S_nk = stirling_number2(n, k)
S_rec = stirling_number2(n-1, k-1) + k * stirling_number2(n-1, k)
print(f"  S(5,3) = {S_nk}")
print(f"  S(4,2) + 3·S(4,3) = {stirling_number2(4,2)} + 3·{stirling_number2(4,3)} = {S_rec}")
print(f"  Match: {S_nk == S_rec}")
print()

print("\n2. GENERATING FUNCTIONS")
print("-" * 60)
print()
print("For Stirling numbers of the first kind:")
print("  x(x-1)(x-2)···(x-n+1) = Σ s(n,k)·x^k (signed version)")
print()

# Demonstrate using symbolic computation
x = var('x')
n = 5
falling_factorial = factorial(x) / factorial(x - n)
expanded = falling_factorial.expand()
print(f"Example: x(x-1)(x-2)(x-3)(x-4) for n={n}")
print(f"  Symbolic form: {falling_factorial}")
print()

# Show coefficients match Stirling numbers (with sign)
print("Coefficients (signed Stirling numbers of first kind):")
for k in range(n+1):
    coeff = (-1)^(n-k) * stirling_number1(n, k)
    print(f"  x^{k}: {coeff:>6} = (-1)^{n-k} · s({n},{k})")
print()

print("For Stirling numbers of the second kind:")
print("  x^n = Σ S(n,k)·x(x-1)(x-2)···(x-k+1)")
print()

print("\n3. BELL NUMBERS")
print("-" * 60)
print()
print("The Bell number B(n) = total number of partitions of n elements")
print("                     = Σ S(n,k) for k=0 to n")
print()
print("  n  |  B(n)  |  Calculation")
print("-" * 60)
for n in range(1, 8):
    bell_n = bell_number(n)
    stirling_sum = sum(stirling_number2(n, k) for k in range(n+1))
    print(f"  {n}  |  {bell_n:>4}  |  {' + '.join(str(stirling_number2(n,k)) for k in range(1,n+1))} = {stirling_sum}")
print()

print("\n4. Total permutations")
print("-" * 60)
print()
print("Stirling numbers appear in many identities. For example:")
print()
print("Identity: n! = Σ s(n,k) (sum over all k)")
print("(Total permutations = sum over all cycle structures)")
print()
print("  n  |   n!   |  Σ s(n,k)")
print("-" * 40)
for n in range(1, 8):
    factorial_n = factorial(n)
    stirling_sum = sum(stirling_number1(n, k) for k in range(n+1))
    print(f"  {n}  |  {factorial_n:>5}  |  {stirling_sum:>5}")
print()

PART 3: KEY PROPERTIES AND RELATIONSHIPS

1. RECURRENCE RELATIONS
------------------------------------------------------------

Stirling numbers of the first kind (unsigned):
  s(n,k) = s(n-1,k-1) + (n-1)·s(n-1,k)

Verification for s(5,3):
  s(5,3) = 35
  s(4,2) + 4·s(4,3) = 11 + 4·6 = 35
  Match: True

Stirling numbers of the second kind:
  S(n,k) = S(n-1,k-1) + k·S(n-1,k)

Verification for S(5,3):
  S(5,3) = 25
  S(4,2) + 3·S(4,3) = 7 + 3·6 = 25
  Match: True


2. GENERATING FUNCTIONS
------------------------------------------------------------

For Stirling numbers of the first kind:
  x(x-1)(x-2)···(x-n+1) = Σ s(n,k)·x^k (signed version)

Example: x(x-1)(x-2)(x-3)(x-4) for n=5
  Symbolic form: factorial(x)/factorial(x - 5)

Coefficients (signed Stirling numbers of first kind):
  x^0:      0 = (-1)^5 · s(5,0)
  x^1:     24 = (-1)^4 · s(5,1)
  x^2:    -50 = (-1)^3 · s(5,2)
  x^3:     35 = (-1)^2 · s(5,3)
  x^4:    -10 = (-1)^1 · s(5,4)
  x^5:      1 = (-1)^0 · s(5,5)

For Stirling nu

In [63]:
# PART 4: Visualization of Growth
print("PART 4: COMPARATIVE GROWTH")
print("Comparing maximum values for each n:")
print()
print("  n  |  max s(n,k)  |  max S(n,k)  |  at k=")
print("-" * 60)
for n in range(1, 10):
    s_values = [stirling_number1(n, k) for k in range(1, n+1)]
    S_values = [stirling_number2(n, k) for k in range(1, n+1)]
    max_s = max(s_values)
    max_S = max(S_values)
    k_max_s = s_values.index(max_s) + 1
    k_max_S = S_values.index(max_S) + 1
    print(f"  {n}  |  {max_s:>10}  |  {max_S:>10}  |  s:{k_max_s}, S:{k_max_S}")

print()
print("=" * 70)
print("Summary:")
print("- Stirling numbers of 1st kind grow from cycle structure of permutations")
print("- Stirling numbers of 2nd kind grow from block structure of set partitions")
print("- Both satisfy elegant recurrence relations")
print("- Both appear throughout combinatorics and number theory")
print("=" * 70)

PART 4: COMPARATIVE GROWTH
Comparing maximum values for each n:

  n  |  max s(n,k)  |  max S(n,k)  |  at k=
------------------------------------------------------------
  1  |           1  |           1  |  s:1, S:1
  2  |           1  |           1  |  s:1, S:1
  3  |           3  |           3  |  s:2, S:2
  4  |          11  |           7  |  s:2, S:2
  5  |          50  |          25  |  s:2, S:3
  6  |         274  |          90  |  s:2, S:3
  7  |        1764  |         350  |  s:2, S:4
  8  |       13132  |        1701  |  s:3, S:4
  9  |      118124  |        7770  |  s:3, S:4

Summary:
- Stirling numbers of 1st kind grow from cycle structure of permutations
- Stirling numbers of 2nd kind grow from block structure of set partitions
- Both satisfy elegant recurrence relations
- Both appear throughout combinatorics and number theory


## Me typed


### Poker 

In [ ]:
############ Poker Simulator ############
Hua=Set(["Hearts","Diamonds","Spades","Clubs"])
Num=Set([2,3,4,5,6,7,8,9,10,"J","Q","K","A"])
Card=cartesian_product([Hua, Num]) # 4*13=52
print(Card)

s(Hua.cardinality())
s(Num.cardinality())
s(Card.cardinality())

s(Card.random_element())                      # pick up a card randomly
Set([Card.random_element(),Card.random_element()]) # two cards randomly

In [ ]:
Hands=Subsets(Card, 5)  # Five-cards game
s(Hands.random_element())
Hands.cardinality()==binomial(52,5)

Subsets(Num,5).cardinality()==binomial(13,5)    # 1287
Flushes=cartesian_product([Subsets(Num,5),Hua]) # 1287*4=5148 
s(Flushes.random_element())
Flushes.cardinality()/Hands.cardinality() # probabilty of the same Hua

### Subsets

In [ ]:
s(binomial(4,2))
A=Subsets([1,2,3,4],2); s(A.cardinality())
s(A.list())
s(A.unrank(1)) # display item (1)
s(A[1]==A.unrank(1))

In [ ]:
A=Set([1,2,3,4]); Nest=Subsets(Subsets(A)) # nested
s(Nest.cardinality())
#s(Nest.list())

### Integer partition & composition & permutation(updating)

In [48]:
P5=Partitions(5);s(P5)
s(P5.cardinality())
P5.list() # compared to Compositions(), order not exist

<IPython.core.display.Math object>

<IPython.core.display.Math object>

[[5], [4, 1], [3, 2], [3, 1, 1], [2, 2, 1], [2, 1, 1, 1], [1, 1, 1, 1, 1]]

In [ ]:
Partitions(999).cardinality()

In [ ]:
P7=Partitions(7)
p=P7.unrank(5)
s(p.ferrers_diagram())
p

In [ ]:
Partition([4,2,1]) # no 's'
WeightedIntegerVectors(8,[2,3,5]).list()

In [50]:
C5=Compositions(5); s(C5.cardinality()); s(C5.list()) # compared to Partitions(), order matters
s([Compositions(n).cardinality() for n in range(10)])
var('x')
s(sum(x^len(c) for c in C5 ))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [53]:
C=IntegerRange(3,21,2); s(C); s(C.cardinality()); s(C.list())

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [55]:
C=Permutations(4); s(C.cardinality()); s(C.list())

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [57]:
C=SetPartitions(["a","b","c"]); s(C.cardinality()); s(C.list())

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### GF 4 enumeration tree

In [ ]:
############ Start from an Integer Seq, to find out GF ############
oeis([1,1,2,5,14])

In [ ]:
# to verify A000108
var('C, z')
# method 1
sys = [C == z + C*C]
s(sys)
sol = solve(sys, C, solution_dict=True) ; #s(sol)
s0=sol[0][C]; s1=sol[1][C]
#s(s0.series(z,6), s1.series(z,6))  # Tylor series, [1][C] is wrong
#C=s0; s(C.series(z,11)); s(C.series(z,101).coefficient(z,100))

# method 2
L.<z>=LazyPowerSeriesRing(QQ)
C=L.undefined(valuation=1)
C.define(z + C*C)
s([C.coefficient(i) for i in range(11)])

z=var('z') # back to closure-type C(z)
C=s0; s(C)
s(derivative(C,z,1))

def d(n): return derivative(s0,n).subs(z=0)
[d(n+1)/d(n) for n in range(1,17)]  # d(n+1)/d(n) = 4n-2 , so C(n+1)/C(n) = (4n-2)/(n+1)

$$c_n = Catalan(n-1) = \frac{1}{n} \binom{2(n-1)}{n-1}$$

In [ ]:
n=var('n')
c=1/n*binomial(2*(n-1),n-1)
s([c.subs(n=k)         for k in range(1,11)])
s([catalan_number(k-1) for k in range(1,11)])